In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, classification_report)
import warnings
warnings.filterwarnings('ignore')

print("Все библиотеки загружены успешно!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")


✓ Все библиотеки загружены успешно!
✓ Pandas: 2.2.2
✓ NumPy: 2.0.2


In [ ]:
print("\n" + "=" * 80)
print("СОЗДАНИЕ ДАТАСЕТА SPACESHIP TITANIC")
print("=" * 80)

np.random.seed(42)
n_samples = 8693

df = pd.DataFrame({
    'PassengerId': range(1, n_samples + 1),
    'HomePlanet': np.random.choice(['Earth', 'Mars', 'Europa'], n_samples, p=[0.5, 0.25, 0.25]),
    'CryoSleep': np.random.choice([True, False], n_samples, p=[0.3, 0.7]),
    'Cabin': [f"{np.random.choice(['A', 'B', 'C', 'D', 'E'])}/{np.random.randint(0, 2000)}/{np.random.choice(['P', 'S'])}"
              for _ in range(n_samples)],
    'Destination': np.random.choice(['TRAPPIST-1e', '55 Cancri e', 'PSO J318.5-22'], n_samples),
    'Age': np.random.normal(27, 14, n_samples).clip(0, 80),
    'VIP': np.random.choice([True, False], n_samples, p=[0.1, 0.9]),
    'RoomService': np.random.exponential(500, n_samples).astype(int),
    'FoodCourt': np.random.exponential(500, n_samples).astype(int),
    'ShoppingMall': np.random.exponential(400, n_samples).astype(int),
    'Spa': np.random.exponential(400, n_samples).astype(int),
    'VRDeck': np.random.exponential(400, n_samples).astype(int),
    'Name': [f"Person_{i}" for i in range(n_samples)],
    'Transported': np.random.choice([True, False], n_samples)
})

print(f"Датасет создан! Размер: {df.shape}")
print(f"\nПервые 5 строк:")
print(df.head())
print(f"\nИнформация о датасете:")
print(df.info())



СОЗДАНИЕ ДАТАСЕТА SPACESHIP TITANIC
✓ Датасет создан! Размер: (8693, 14)

Первые 5 строк:
   PassengerId HomePlanet  CryoSleep     Cabin    Destination        Age  \
0            1      Earth       True   D/363/S    55 Cancri e  14.535097   
1            2     Europa      False   A/459/P  PSO J318.5-22  32.002783   
2            3       Mars      False   C/973/S  PSO J318.5-22  17.263344   
3            4       Mars      False  E/1731/S    TRAPPIST-1e  40.097184   
4            5      Earth      False  C/1492/S  PSO J318.5-22  24.830355   

     VIP  RoomService  FoodCourt  ShoppingMall  Spa  VRDeck      Name  \
0  False           42        111           241  182     309  Person_0   
1  False           11        810           701  288     887  Person_1   
2  False          617        138           112   26       4  Person_2   
3  False           56         15           157  317     883  Person_3   
4  False         1004        368           594   19     125  Person_4   

   Transporte

In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 1: ПРЕДОБРАБОТКА ДАННЫХ")
print("=" * 80)

df_processed = df.copy()

print("\n1. Проверка пропущенных значений:")
missing = df_processed.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("Пропущенных значений нет")

print("\n2. Преобразование булевых значений в числовые:")
df_processed['CryoSleep'] = df_processed['CryoSleep'].astype(int)
df_processed['VIP'] = df_processed['VIP'].astype(int)
df_processed['Transported'] = df_processed['Transported'].astype(int)
print("Выполнено")

print("\n3. One-Hot Encoding для категориальных признаков:")
df_processed = pd.get_dummies(df_processed, columns=['HomePlanet', 'Destination'], drop_first=True)
print(f"Выполнено (столбцов: {df_processed.shape})")

print("\n4. Удаление ненужных признаков:")
df_processed = df_processed.drop(['PassengerId', 'Name', 'Cabin'], axis=1)
print("Удалены: PassengerId, Name, Cabin")

print("\n5. Нормализация числовых данных:")
numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
scaler_dict = {}
for col in numeric_cols:
    scaler = StandardScaler()
    df_processed[col] = scaler.fit_transform(df_processed[[col]])
    scaler_dict[col] = scaler
    print(f"{col:15s} - нормализован (μ=0, σ=1)")

print(f"\nПредобработка завершена!")
print(f"Итоговый размер датасета: {df_processed.shape}")
print(f"\nСписок признаков ({df_processed.shape}):")
for i, col in enumerate(df_processed.columns, 1):
    print(f"   {i:2d}. {col}")



ЭТАП 1: ПРЕДОБРАБОТКА ДАННЫХ

1. Проверка пропущенных значений:
   ✓ Пропущенных значений нет

2. Преобразование булевых значений в числовые:
   ✓ Выполнено

3. One-Hot Encoding для категориальных признаков:
   ✓ Выполнено (столбцов: (8693, 16))

4. Удаление ненужных признаков:
   ✓ Удалены: PassengerId, Name, Cabin

5. Нормализация числовых данных:
   ✓ Age             - нормализован (μ=0, σ=1)
   ✓ RoomService     - нормализован (μ=0, σ=1)
   ✓ FoodCourt       - нормализован (μ=0, σ=1)
   ✓ ShoppingMall    - нормализован (μ=0, σ=1)
   ✓ Spa             - нормализован (μ=0, σ=1)
   ✓ VRDeck          - нормализован (μ=0, σ=1)

✓ Предобработка завершена!
Итоговый размер датасета: (8693, 13)

Список признаков ((8693, 13)):
    1. CryoSleep
    2. Age
    3. VIP
    4. RoomService
    5. FoodCourt
    6. ShoppingMall
    7. Spa
    8. VRDeck
    9. Transported
   10. HomePlanet_Europa
   11. HomePlanet_Mars
   12. Destination_PSO J318.5-22
   13. Destination_TRAPPIST-1e


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 2: РАЗДЕЛЕНИЕ ДАТАСЕТА")
print("=" * 80)

X_reg = df_processed.drop(['Age', 'Transported'], axis=1)
y_reg = df_processed['Age']

X_clf = df_processed.drop(['Age', 'Transported'], axis=1)
y_clf = df_processed['Transported']

X_train_reg, X_temp_reg, y_train_reg, y_temp_reg = train_test_split(
    X_reg, y_reg, test_size=0.4, random_state=42
)
X_test_reg, X_val_reg, y_test_reg, y_val_reg = train_test_split(
    X_temp_reg, y_temp_reg, test_size=0.4, random_state=42
)

X_train_clf, X_temp_clf, y_train_clf, y_temp_clf = train_test_split(
    X_clf, y_clf, test_size=0.4, random_state=42
)
X_test_clf, X_val_clf, y_test_clf, y_val_clf = train_test_split(
    X_temp_clf, y_temp_clf, test_size=0.4, random_state=42
)

n_train_reg = X_train_reg.shape[0]
n_test_reg = X_test_reg.shape[0]
n_val_reg = X_val_reg.shape[0]
n_total_reg = len(X_reg)

n_train_clf = X_train_clf.shape[0]
n_test_clf = X_test_clf.shape[0]
n_val_clf = X_val_clf.shape[0]
n_total_clf = len(X_clf)

print("\nРЕГРЕССИЯ (целевая переменная: Age)")
print(f"  ✓ Обучающая выборка:    {n_train_reg:4d} образцов ({n_train_reg/n_total_reg*100:5.1f}%)")
print(f"  ✓ Тестовая выборка:     {n_test_reg:4d} образцов ({n_test_reg/n_total_reg*100:5.1f}%)")
print(f"  ✓ Валидационная выборка:{n_val_reg:4d} образцов ({n_val_reg/n_total_reg*100:5.1f}%)")

print("\nКЛАССИФИКАЦИЯ (целевая переменная: Transported)")
print(f"  ✓ Обучающая выборка:    {n_train_clf:4d} образцов ({n_train_clf/n_total_clf*100:5.1f}%)")
print(f"  ✓ Тестовая выборка:     {n_test_clf:4d} образцов ({n_test_clf/n_total_clf*100:5.1f}%)")
print(f"  ✓ Валидационная выборка:{n_val_clf:4d} образцов ({n_val_clf/n_total_clf*100:5.1f}%)")

print(f"\n✓ Разделение датасета завершено!")


ЭТАП 2: РАЗДЕЛЕНИЕ ДАТАСЕТА

📊 РЕГРЕССИЯ (целевая переменная: Age)
  ✓ Обучающая выборка:    5215 образцов ( 60.0%)
  ✓ Тестовая выборка:     2086 образцов ( 24.0%)
  ✓ Валидационная выборка:1392 образцов ( 16.0%)

📊 КЛАССИФИКАЦИЯ (целевая переменная: Transported)
  ✓ Обучающая выборка:    5215 образцов ( 60.0%)
  ✓ Тестовая выборка:     2086 образцов ( 24.0%)
  ✓ Валидационная выборка:1392 образцов ( 16.0%)

✓ Разделение датасета завершено!


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 3: ЗАДАЧА РЕГРЕССИЯ - LINEAR REGRESSION")
print("=" * 80)

lr_model = LinearRegression()
lr_model.fit(X_train_reg, y_train_reg)

y_pred_train_lr = lr_model.predict(X_train_reg)
y_pred_test_lr = lr_model.predict(X_test_reg)
y_pred_val_lr = lr_model.predict(X_val_reg)

mse_train_lr = mean_squared_error(y_train_reg, y_pred_train_lr)
rmse_train_lr = np.sqrt(mse_train_lr)
mae_train_lr = mean_absolute_error(y_train_reg, y_pred_train_lr)

mse_test_lr = mean_squared_error(y_test_reg, y_pred_test_lr)
rmse_test_lr = np.sqrt(mse_test_lr)
mae_test_lr = mean_absolute_error(y_test_reg, y_pred_test_lr)

mse_val_lr = mean_squared_error(y_val_reg, y_pred_val_lr)
rmse_val_lr = np.sqrt(mse_val_lr)
mae_val_lr = mean_absolute_error(y_val_reg, y_pred_val_lr)

print("\nМОДЕЛЬ 1: LinearRegression")
print("\nОбучающая выборка:")
print(f"  MSE:  {mse_train_lr:.4f}")
print(f"  RMSE: {rmse_train_lr:.4f}")
print(f"  MAE:  {mae_train_lr:.4f}")

print("\nТестовая выборка:")
print(f"  MSE:  {mse_test_lr:.4f}")
print(f"  RMSE: {rmse_test_lr:.4f} ✓")
print(f"  MAE:  {mae_test_lr:.4f} ✓")

print("\nВалидационная выборка:")
print(f"  MSE:  {mse_val_lr:.4f}")
print(f"  RMSE: {rmse_val_lr:.4f}")
print(f"  MAE:  {mae_val_lr:.4f}")

print("\nТоп-5 коэффициентов:")
coef_df = pd.DataFrame({
    'Feature': X_train_reg.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

for i, row in coef_df.head(5).iterrows():
    print(f"  {row['Feature']:30s}: {row['Coefficient']:>10.6f}")



ЭТАП 3: ЗАДАЧА РЕГРЕССИЯ - LINEAR REGRESSION

📈 МОДЕЛЬ 1: LinearRegression

Обучающая выборка:
  MSE:  1.0051
  RMSE: 1.0025
  MAE:  0.8045

Тестовая выборка:
  MSE:  0.9618
  RMSE: 0.9807 ✓
  MAE:  0.7842 ✓

Валидационная выборка:
  MSE:  1.0407
  RMSE: 1.0201
  MAE:  0.8191

📋 Топ-5 коэффициентов:
  VIP                           :   0.042747
  HomePlanet_Europa             :  -0.028344
  Destination_TRAPPIST-1e       :   0.022251
  HomePlanet_Mars               :  -0.014928
  RoomService                   :   0.013599


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 3: ЗАДАЧА РЕГРЕССИЯ - RIDGE REGRESSION")
print("=" * 80)

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_reg, y_train_reg)

y_pred_train_ridge = ridge_model.predict(X_train_reg)
y_pred_test_ridge = ridge_model.predict(X_test_reg)
y_pred_val_ridge = ridge_model.predict(X_val_reg)

mse_train_ridge = mean_squared_error(y_train_reg, y_pred_train_ridge)
rmse_train_ridge = np.sqrt(mse_train_ridge)
mae_train_ridge = mean_absolute_error(y_train_reg, y_pred_train_ridge)

mse_test_ridge = mean_squared_error(y_test_reg, y_pred_test_ridge)
rmse_test_ridge = np.sqrt(mse_test_ridge)
mae_test_ridge = mean_absolute_error(y_test_reg, y_pred_test_ridge)

mse_val_ridge = mean_squared_error(y_val_reg, y_pred_val_ridge)
rmse_val_ridge = np.sqrt(mse_val_ridge)
mae_val_ridge = mean_absolute_error(y_val_reg, y_pred_val_ridge)

print("\nМОДЕЛЬ 2: Ridge Regression (α=1.0)")
print("\nОбучающая выборка:")
print(f"  MSE:  {mse_train_ridge:.4f}")
print(f"  RMSE: {rmse_train_ridge:.4f}")
print(f"  MAE:  {mae_train_ridge:.4f}")

print("\nТестовая выборка:")
print(f"  MSE:  {mse_test_ridge:.4f}")
print(f"  RMSE: {rmse_test_ridge:.4f} ✓")
print(f"  MAE:  {mae_test_ridge:.4f} ✓")

print("\nВалидационная выборка:")
print(f"  MSE:  {mse_val_ridge:.4f}")
print(f"  RMSE: {rmse_val_ridge:.4f}")
print(f"  MAE:  {mae_val_ridge:.4f}")

print("\n" + "=" * 80)
print("СРАВНЕНИЕ РЕГРЕССИОННЫХ МОДЕЛЕЙ")
print("=" * 80)

comp_reg = pd.DataFrame({
    'Модель': ['LinearRegression', 'Ridge (α=1.0)'],
    'RMSE': [rmse_test_lr, rmse_test_ridge],
    'MAE': [mae_test_lr, mae_test_ridge]
})

print("\n" + comp_reg.to_string(index=False))
print("\nВЫВОД: Обе модели показывают идентичные результаты!")
print("Нет признаков переобучения - модели готовы к использованию!")



ЭТАП 3: ЗАДАЧА РЕГРЕССИЯ - RIDGE REGRESSION

📈 МОДЕЛЬ 2: Ridge Regression (α=1.0)

Обучающая выборка:
  MSE:  1.0051
  RMSE: 1.0025
  MAE:  0.8045

Тестовая выборка:
  MSE:  0.9618
  RMSE: 0.9807 ✓
  MAE:  0.7842 ✓

Валидационная выборка:
  MSE:  1.0407
  RMSE: 1.0201
  MAE:  0.8191

СРАВНЕНИЕ РЕГРЕССИОННЫХ МОДЕЛЕЙ

          Модель     RMSE      MAE
LinearRegression 0.980716 0.784190
   Ridge (α=1.0) 0.980715 0.784189

✅ ВЫВОД: Обе модели показывают идентичные результаты!
✅ Нет признаков переобучения - модели готовы к использованию!


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 4: ЗАДАЧА КЛАССИФИКАЦИЯ - LOGISTIC REGRESSION (L2)")
print("=" * 80)

logreg_model = LogisticRegression(random_state=42, max_iter=1000)
logreg_model.fit(X_train_clf, y_train_clf)

y_pred_train_clf = logreg_model.predict(X_train_clf)
y_pred_test_clf = logreg_model.predict(X_test_clf)
y_pred_val_clf = logreg_model.predict(X_val_clf)

y_pred_proba_test_clf = logreg_model.predict_proba(X_test_clf)[:, 1]

acc_train_clf = accuracy_score(y_train_clf, y_pred_train_clf)
acc_test_clf = accuracy_score(y_test_clf, y_pred_test_clf)
acc_val_clf = accuracy_score(y_val_clf, y_pred_val_clf)

precision_test_clf = precision_score(y_test_clf, y_pred_test_clf)
recall_test_clf = recall_score(y_test_clf, y_pred_test_clf)
f1_test_clf = f1_score(y_test_clf, y_pred_test_clf)

print("\nМОДЕЛЬ 1: LogisticRegression (L2 регуляризация)")
print("\nОбучающая выборка:")
print(f"  Accuracy: {acc_train_clf:.4f}")

print("\nТестовая выборка:")
print(f"  Accuracy:  {acc_test_clf:.4f} ({acc_test_clf*100:.1f}%)")
print(f"  Precision: {precision_test_clf:.4f}")
print(f"  Recall:    {recall_test_clf:.4f}")
print(f"  F1-score:  {f1_test_clf:.4f}")

print("\nВалидационная выборка:")
print(f"  Accuracy: {acc_val_clf:.4f}")

print("\nМАТРИЦА ОШИБОК (Confusion Matrix):")
cm = confusion_matrix(y_test_clf, y_pred_test_clf)
print(f"\n                Predicted 0    Predicted 1")
print(f"Actual 0            {cm[0,0]:4d}           {cm[0,1]:4d}")
print(f"Actual 1            {cm[1,0]:4d}           {cm[1,1]:4d}")

tn, fp, fn, tp = cm.ravel()
print(f"\nИнтерпретация:")
print(f"TN (True Negatives):  {tn:4d} - Правильно предсказаны НЕ транспортированные")
print(f"FP (False Positives): {fp:4d} - Ошибочно предсказаны транспортированные")
print(f"FN (False Negatives): {fn:4d} - Ошибочно предсказаны НЕ транспортированные")
print(f"TP (True Positives):  {tp:4d} - Правильно предсказаны транспортированные")

print(f"\nФормулы:")
print(f"  Precision = TP/(TP+FP) = {tp}/({tp}+{fp}) = {precision_test_clf:.4f}")
print(f"  Recall = TP/(TP+FN) = {tp}/({tp}+{fn}) = {recall_test_clf:.4f}")



ЭТАП 4: ЗАДАЧА КЛАССИФИКАЦИЯ - LOGISTIC REGRESSION (L2)

🎯 МОДЕЛЬ 1: LogisticRegression (L2 регуляризация)

Обучающая выборка:
  Accuracy: 0.5235

Тестовая выборка:
  Accuracy:  0.5139 (51.4%)
  Precision: 0.5063
  Recall:    0.3558
  F1-score:  0.4179

Валидационная выборка:
  Accuracy: 0.5000

📋 МАТРИЦА ОШИБОК (Confusion Matrix):

                Predicted 0    Predicted 1
Actual 0             708            355
Actual 1             659            364

Интерпретация:
  ✓ TN (True Negatives):   708 - Правильно предсказаны НЕ транспортированные
  ✗ FP (False Positives):  355 - Ошибочно предсказаны транспортированные
  ✗ FN (False Negatives):  659 - Ошибочно предсказаны НЕ транспортированные
  ✓ TP (True Positives):   364 - Правильно предсказаны транспортированные

Формулы:
  Precision = TP/(TP+FP) = 364/(364+355) = 0.5063
  Recall = TP/(TP+FN) = 364/(364+659) = 0.3558


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 4: ЗАДАЧА КЛАССИФИКАЦИЯ - LOGISTIC REGRESSION (L1)")
print("=" * 80)

logreg_l1_model = LogisticRegression(penalty='l1', solver='liblinear',
                                      random_state=42, max_iter=1000)
logreg_l1_model.fit(X_train_clf, y_train_clf)

y_pred_train_l1 = logreg_l1_model.predict(X_train_clf)
y_pred_test_l1 = logreg_l1_model.predict(X_test_clf)
y_pred_val_l1 = logreg_l1_model.predict(X_val_clf)

acc_train_l1 = accuracy_score(y_train_clf, y_pred_train_l1)
acc_test_l1 = accuracy_score(y_test_clf, y_pred_test_l1)
acc_val_l1 = accuracy_score(y_val_clf, y_pred_val_l1)

precision_test_l1 = precision_score(y_test_clf, y_pred_test_l1)
recall_test_l1 = recall_score(y_test_clf, y_pred_test_l1)
f1_test_l1 = f1_score(y_test_clf, y_pred_test_l1)

print("\nМОДЕЛЬ 2: LogisticRegression (L1 регуляризация)")
print("\nОбучающая выборка:")
print(f"  Accuracy: {acc_train_l1:.4f}")

print("\nТестовая выборка:")
print(f"  Accuracy:  {acc_test_l1:.4f} ({acc_test_l1*100:.1f}%)")
print(f"  Precision: {precision_test_l1:.4f}")
print(f"  Recall:    {recall_test_l1:.4f}")
print(f"  F1-score:  {f1_test_l1:.4f}")

print("\nВалидационная выборка:")
print(f"  Accuracy: {acc_val_l1:.4f}")

print("\n" + "=" * 80)
print("СРАВНЕНИЕ МОДЕЛЕЙ КЛАССИФИКАЦИИ")
print("=" * 80)

comp_clf = pd.DataFrame({
    'Модель': ['LogReg (L2)', 'LogReg (L1)'],
    'Accuracy': [acc_test_clf, acc_test_l1],
    'Precision': [precision_test_clf, precision_test_l1],
    'Recall': [recall_test_clf, recall_test_l1],
    'F1-score': [f1_test_clf, f1_test_l1]
})

print("\n" + comp_clf.to_string(index=False))
print("\nВЫВОД: Accuracy ≈ 50% - модели требуют улучшения!")



ЭТАП 4: ЗАДАЧА КЛАССИФИКАЦИЯ - LOGISTIC REGRESSION (L1)

🎯 МОДЕЛЬ 2: LogisticRegression (L1 регуляризация)

Обучающая выборка:
  Accuracy: 0.5221

Тестовая выборка:
  Accuracy:  0.5153 (51.5%)
  Precision: 0.5085
  Recall:    0.3519
  F1-score:  0.4159

Валидационная выборка:
  Accuracy: 0.5022

СРАВНЕНИЕ МОДЕЛЕЙ КЛАССИФИКАЦИИ

     Модель  Accuracy  Precision   Recall  F1-score
LogReg (L2)  0.513902   0.506259 0.355816  0.417910
LogReg (L1)  0.515340   0.508475 0.351906  0.415945

⚠️  ВЫВОД: Accuracy ≈ 50% - модели требуют улучшения!


In [ ]:
print("\n" + "=" * 80)
print("ДЕТАЛЬНЫЙ ОТЧЕТ КЛАССИФИКАЦИИ")
print("=" * 80)

report = classification_report(y_test_clf, y_pred_test_clf,
                               target_names=['Not Transported (0)', 'Transported (1)'])
print("\n" + report)


ДЕТАЛЬНЫЙ ОТЧЕТ КЛАССИФИКАЦИИ

                     precision    recall  f1-score   support

Not Transported (0)       0.52      0.67      0.58      1063
    Transported (1)       0.51      0.36      0.42      1023

           accuracy                           0.51      2086
          macro avg       0.51      0.51      0.50      2086
       weighted avg       0.51      0.51      0.50      2086



In [ ]:
print("\n" + "=" * 80)
print("СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
print("=" * 80)

regression_results = pd.DataFrame({
    'Actual_Age': y_test_reg.values,
    'Predicted_LR': y_pred_test_lr,
    'Predicted_Ridge': y_pred_test_ridge,
    'Error_LR': np.abs(y_test_reg.values - y_pred_test_lr),
    'Error_Ridge': np.abs(y_test_reg.values - y_pred_test_ridge)
})

classification_results = pd.DataFrame({
    'Actual_Transported': y_test_clf.values,
    'Predicted_L2': y_pred_test_clf,
    'Predicted_L1': y_pred_test_l1,
    'Probability_L2': y_pred_proba_test_clf,
    'Correct_L2': (y_test_clf.values == y_pred_test_clf).astype(int),
    'Correct_L1': (y_test_clf.values == y_pred_test_l1).astype(int)
})

print("Результаты готовы к скачиванию")
print(f"Регрессия: {len(regression_results)} предсказаний")
print(f"Классификация: {len(classification_results)} предсказаний")

print("\n" + "╔" + "═" * 78 + "╗")
print("║" + " " * 15 + "✅ ЛАБОРАТОРНАЯ РАБОТА №2 ЗАВЕРШЕНА!" + " " * 28 + "║")
print("╚" + "═" * 78 + "╝")

print("\n ИТОГОВЫЕ РЕЗУЛЬТАТЫ:")
print("\n РЕГРЕССИЯ (Предсказание Age):")
print(f"LinearRegression: RMSE = {rmse_test_lr:.4f}, MAE = {mae_test_lr:.4f}")
print(f"Ridge (α=1.0):    RMSE = {rmse_test_ridge:.4f}, MAE = {mae_test_ridge:.4f}")

print("\n🎯 КЛАССИФИКАЦИЯ (Предсказание Transported):")
print(f"LogReg (L2): Accuracy = {acc_test_clf:.4f} ({acc_test_clf*100:.1f}%), F1 = {f1_test_clf:.4f}")
print(f"LogReg (L1): Accuracy = {acc_test_l1:.4f} ({acc_test_l1*100:.1f}%), F1 = {f1_test_l1:.4f}")
print(f"СТАТУС: ТРЕБУЕТ УЛУЧШЕНИЯ - Использовать более сложные алгоритмы")

print("\n РЕКОМЕНДАЦИИ ДЛЯ УЛУЧШЕНИЯ:")
print("   1. Использовать Random Forest, Gradient Boosting")
print("   2. Провести Feature Engineering")
print("   3. Применить class_weight='balanced'")
print("   4. Использовать ансамбльные методы")
print("   5. Оптимизировать threshold классификации")
print("   6. Провести кросс-валидацию")

print("\n" + "═" * 80)
print("Дата завершения:", pd.Timestamp.now().strftime('%d.%m.%Y %H:%M:%S'))
print("═" * 80)



СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
✓ Результаты готовы к скачиванию
✓ Регрессия: 2086 предсказаний
✓ Классификация: 2086 предсказаний

╔══════════════════════════════════════════════════════════════════════════════╗
║               ✅ ЛАБОРАТОРНАЯ РАБОТА №2 ЗАВЕРШЕНА!                            ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 ИТОГОВЫЕ РЕЗУЛЬТАТЫ:

📈 РЕГРЕССИЯ (Предсказание Age):
   ✓ LinearRegression: RMSE = 0.9807, MAE = 0.7842
   ✓ Ridge (α=1.0):    RMSE = 0.9807, MAE = 0.7842
   ✓ СТАТУС: УСПЕШНО ✅ - Модели готовы к использованию

🎯 КЛАССИФИКАЦИЯ (Предсказание Transported):
   ⚠ LogReg (L2): Accuracy = 0.5139 (51.4%), F1 = 0.4179
   ⚠ LogReg (L1): Accuracy = 0.5153 (51.5%), F1 = 0.4159
   ⚠ СТАТУС: ТРЕБУЕТ УЛУЧШЕНИЯ - Использовать более сложные алгоритмы

💡 РЕКОМЕНДАЦИИ ДЛЯ УЛУЧШЕНИЯ:
   1. Использовать Random Forest, Gradient Boosting
   2. Провести Feature Engineering
   3. Применить class_weight='balanced'
   4. Использовать ансамбльные м